# Probabilistic Interpretation of Linear Regression

This notebook explores linear regression from a probabilistic perspective, connecting the classical optimization approach to statistical inference.

## Important Note on Mathematical Notation

Throughout this notebook, whenever we use a mathematical function or formula (such as probability density functions, likelihood functions, Bayes' rule, or matrix operations), we first present the **general foundational equation** before applying it to our specific regression problem. This approach ensures clarity by:

1. Introducing the general concept and formula
2. Explaining what each component means
3. Showing how we apply it to linear regression

This makes it easier to understand not just *how* we use each formula, but *why* it works and where it comes from.

## Learning Objectives
By the end of this notebook, you will be able to:
1. **Derive** the Gaussian likelihood $p(y \mid x, \theta, \sigma^2) = \mathcal{N}(\theta^T x, \sigma^2)$ from the noise model $y = \theta^T x + \epsilon$.
2. **Show** that maximizing the log-likelihood under Gaussian noise is equivalent to minimizing the sum of squared errors.
3. **Compute** the MLE for both $\theta$ and $\sigma^2$, arriving at the Normal Equation $\hat{\theta} = (X^TX)^{-1}X^Ty$.
4. **Interpret** prediction uncertainty as the sum of aleatoric (irreducible noise) and epistemic (parameter) uncertainty.
5. **Apply** Bayes' rule with a Gaussian prior to derive the posterior $p(\theta \mid X, y) = \mathcal{N}(\mu_n, \Sigma_n)$.
6. **Explain** why MLE is the limiting case of Bayesian inference with a diffuse prior.

> **Prerequisite**: Gaussian distribution, MLE, least squares.

## 1. Introduction to Probabilistic Linear Regression

In the classical machine learning view, linear regression is an optimization problem:

$$\min_\theta \sum_{i=1}^{n} (y_i - \theta^T x_i)^2$$

### Understanding the Formula

Let's break down what each symbol means:

- **$n$**: The number of training samples or observations
- **$i$**: An index that ranges from 1 to $n$, referring to the $i$-th sample
- **$y_i$**: The target value (output, dependent variable) for the $i$-th sample. This is what we're trying to predict. For example, if predicting house prices, this would be the price of house $i$.
- **$x_i$**: The feature vector (input, independent variables) for the $i$-th sample. This is typically a column vector containing all the features for that sample. For example, $x_i = [x_{i,1}, x_{i,2}, \ldots, x_{i,p}]^T$ where $p$ is the number of features.
- **$\theta$**: The parameter vector we want to learn (weights/coefficients). Shape: $(p \times 1)$. These are the unknowns we're solving for.
- **$\theta^T x_i$**: The dot product (inner product) between $\theta$ and $x_i$, producing a scalar prediction. This represents the model's predicted value for sample $i$.
- **$y_i - \theta^T x_i$**: The residual or error - the difference between the actual value and predicted value for sample $i$
- **$(y_i - \theta^T x_i)^2$**: The squared error for sample $i$
- **$\sum_{i=1}^{n}$**: The sum over all $n$ samples - total squared error across all training data

### Example

Suppose we have 3 houses with 2 features each (size and age):

| Sample | Size ($x_{i,1}$) | Age ($x_{i,2}$) | Price ($y_i$) |
|--------|------------------|-----------------|---------------|
| 1 | 2000 | 5 | 300,000 |
| 2 | 1500 | 10 | 250,000 |
| 3 | 2500 | 2 | 350,000 |

For sample 1: $x_1 = [2000, 5]^T$ and $y_1 = 300,000$

If $\theta = [100, -1000]^T$ (hypothetical weights), then:
- Prediction: $\theta^T x_1 = 100 \times 2000 + (-1000) \times 5 = 200,000 - 5,000 = 195,000$
- Residual: $y_1 - \theta^T x_1 = 300,000 - 195,000 = 105,000$
- Squared error: $(105,000)^2 = 11,025,000,000$

The goal is to find $\theta$ that minimizes the total squared error across all samples.

### Probabilistic View

However, we can interpret linear regression probabilistically by modeling the relationship between features $x$ and target $y$ as a statistical process.

## 2. The Probabilistic Model

### 2.1 Model Assumption

We assume the target variable $y$ is generated by a linear function of the input features plus Gaussian noise:

$$y = \theta^T x + \epsilon$$

where:
- $y$: The observed target value
- $\theta^T x$: The linear combination of parameters $\theta = [\theta_1, \theta_2, \ldots, \theta_p]^T$ and features $x = [x_1, x_2, \ldots, x_p]^T$, computed as $\theta_1 x_1 + \theta_2 x_2 + \cdots + \theta_p x_p$
- $\epsilon \sim \mathcal{N}(0, \sigma^2)$: Gaussian noise with mean 0 and variance $\sigma^2$ (the irreducible randomness in the data)

### 2.2 Distribution of Output

This means $y$ conditioned on $x$ follows a Gaussian distribution:

$$p(y | x, \theta, \sigma^2) = \mathcal{N}(\theta^T x, \sigma^2)$$

Explanation:
- $p(y | x, \theta, \sigma^2)$: The probability density of observing $y$ given features $x$, parameters $\theta$, and noise variance $\sigma^2$
- $\mathcal{N}(\theta^T x, \sigma^2)$: A normal (Gaussian) distribution with mean $\mu = \theta^T x$ and variance $\sigma^2$
- This says: for a given $x$, the target $y$ is normally distributed around the prediction $\theta^T x$ with spread determined by $\sigma^2$

#### General Gaussian Probability Density Function

We know that the probability density function (PDF) for a Gaussian distribution with mean $\mu$ and variance $\sigma^2$ is given by:

$$p(z) = \frac{1}{\sqrt{2\pi\sigma^2}} \exp\left(-\frac{(z - \mu)^2}{2\sigma^2}\right)$$

where:
- $z$: The random variable
- $\mu$: Mean of the distribution (center of the bell curve)
- $\sigma^2$: Variance (controls the width/spread)
- $\frac{1}{\sqrt{2\pi\sigma^2}}$: Normalization constant that ensures the PDF integrates to 1
- $\exp(\cdot)$: The exponential function

#### Application to Our Regression Model

Applying this general formula to our case where $z = y$, $\mu = \theta^T x$, we get:

$$p(y | x, \theta, \sigma^2) = \frac{1}{\sqrt{2\pi\sigma^2}} \exp\left(-\frac{(y - \theta^T x)^2}{2\sigma^2}\right)$$

Component breakdown:
- The numerator inside the exponent $(y - \theta^T x)^2$ is the squared deviation from the predicted mean
- $2\sigma^2$ in the denominator of the exponent controls how quickly the probability decreases with error (larger $\sigma^2$ = wider, flatter curve)
- $\frac{1}{\sqrt{2\pi\sigma^2}}$ is the normalization constant ensuring the probability integrates to 1

## 3. Maximum Likelihood Estimation

### 3.1 Likelihood Function

#### Understanding the Likelihood Concept

We know that the **likelihood function** is a measure of how probable the observed data is, given a specific set of parameters. It's constructed from the probability density function by multiplying the individual probabilities of each observation:

$$\text{Likelihood}(\theta | \text{data}) = \prod_{i=1}^{n} p(\text{observation}_i | \theta)$$

For our regression model with samples $(x_1, y_1), (x_2, y_2), \ldots, (x_n, y_n)$, we use the Gaussian PDF from section 2.2 for each sample:

$$L(\theta, \sigma^2) = \prod_{i=1}^{n} p(y_i | x_i, \theta, \sigma^2)$$

where:
- $\prod_{i=1}^{n}$: Multiply the probability for each of the $n$ samples (because samples are independent)
- $p(y_i | x_i, \theta, \sigma^2)$: Probability of observing the specific $y_i$ value given its features $x_i$ and our parameters

#### Expanded Form

Substituting our Gaussian PDF into the likelihood:

$$L(\theta, \sigma^2) = \prod_{i=1}^{n} \frac{1}{\sqrt{2\pi\sigma^2}} \exp\left(-\frac{(y_i - \theta^T x_i)^2}{2\sigma^2}\right)$$

where each factor represents how well our model predicts the $i$-th sample:
- If the prediction $\theta^T x_i$ is close to $y_i$, this factor is large (close to $\frac{1}{\sqrt{2\pi\sigma^2}}$)
- If the prediction is far from $y_i$, this factor is small (due to the negative exponent)

**Interpretation**: The likelihood is the probability we would observe this exact data if our parameters were correct. Higher likelihood means our parameters explain the data better.

### 3.2 Log-Likelihood

#### Why Take the Logarithm?

Working with the likelihood directly is numerically unstable (multiplying many small probabilities). We know that the **log-likelihood** is simply the natural logarithm of the likelihood function:

$$\ell(\theta, \sigma^2) = \log L(\theta, \sigma^2)$$

This is mathematically equivalent because the logarithm is a monotonically increasing function—maximizing $L$ is the same as maximizing $\log L$. The advantages are:
- Products become sums (easier to compute)
- Exponentials become linear (easier to differentiate)
- Better numerical stability

#### Deriving the Log-Likelihood

Starting with the likelihood:

$$L(\theta, \sigma^2) = \prod_{i=1}^{n} \frac{1}{\sqrt{2\pi\sigma^2}} \exp\left(-\frac{(y_i - \theta^T x_i)^2}{2\sigma^2}\right)$$

Taking the natural logarithm of both sides:

$$\ell(\theta, \sigma^2) = \log\left(\prod_{i=1}^{n} \frac{1}{\sqrt{2\pi\sigma^2}} \exp\left(-\frac{(y_i - \theta^T x_i)^2}{2\sigma^2}\right)\right)$$

Using logarithm properties ($\log(AB) = \log A + \log B$ and $\log(e^x) = x$):

$$\ell(\theta, \sigma^2) = \sum_{i=1}^{n} \left(\log\frac{1}{\sqrt{2\pi\sigma^2}} + \left(-\frac{(y_i - \theta^T x_i)^2}{2\sigma^2}\right)\right)$$

$$= \sum_{i=1}^{n} \left(-\frac{1}{2}\log(2\pi\sigma^2) - \frac{(y_i - \theta^T x_i)^2}{2\sigma^2}\right)$$

$$= -\frac{n}{2}\log(2\pi\sigma^2) - \frac{1}{2\sigma^2}\sum_{i=1}^{n}(y_i - \theta^T x_i)^2$$

Component breakdown:
- $-\frac{n}{2}\log(2\pi\sigma^2)$: Penalty term that depends on the noise level; larger $\sigma^2$ makes this term smaller (more permissive)
- $\frac{1}{2\sigma^2}\sum_{i=1}^{n}(y_i - \theta^T x_i)^2$: Sum of squared errors, scaled by the inverse of noise variance
- The sum $\sum_{i=1}^{n}$ aggregates errors across all $n$ samples

### 3.3 Maximum Likelihood Estimates

#### Finding the Maximum

We know that to find the **maximum** of a function, we use calculus: we take the derivative and set it equal to zero. At a maximum, the function stops increasing and starts decreasing, so the slope (derivative) is zero.

For our log-likelihood function $\ell(\theta, \sigma^2)$, we want to find the values of $\theta$ that maximize it.

#### Taking the Derivative

We take the **partial derivative** of the log-likelihood with respect to $\theta$ (treating $\sigma^2$ as a constant) and set it to zero:

$$\frac{\partial \ell}{\partial \theta} = -\frac{1}{\sigma^2}\sum_{i=1}^{n}(y_i - \theta^T x_i)(-x_i) = 0$$

Breaking down this derivative:
- $\frac{\partial \ell}{\partial \theta}$: The partial derivative—how the log-likelihood changes as we change $\theta$
- $(y_i - \theta^T x_i)$: The residual (prediction error) for sample $i$ from the $-\frac{(y_i - \theta^T x_i)^2}{2\sigma^2}$ term
- $(-x_i)$: The chain rule from calculus gives us $-x_i$ when differentiating $\theta^T x_i$ with respect to $\theta$
- The sum $\sum_{i=1}^{n}$ aggregates across all samples

#### Simplifying the Equation

Multiplying both sides by $-\sigma^2$ (which is positive, so doesn't change the equation):

$$\sum_{i=1}^{n} x_i (y_i - \theta^T x_i) = 0$$

This elegant result says: **the sum of "feature times residual" must equal zero**. In other words, the errors must be orthogonal (perpendicular) to the features. This is known as the **normal equation** in least squares terminology.

#### Matrix Form (Compact Notation)

Define:
- $X$: $n \times p$ design matrix (each row is one sample's features)
- $y$: $n \times 1$ target vector (each element is one sample's target)
- $\theta$: $p \times 1$ parameter vector (the weights we're solving for)

Then the normal equation becomes:
$$X^T(y - X\theta) = 0$$

where:
- $X\theta$: $n \times 1$ vector of predictions for all samples
- $(y - X\theta)$: $n \times 1$ vector of residuals for all samples
- $X^T$: Transpose of $X$, shape $p \times n$
- $X^T(y - X\theta) = 0$ represents $p$ equations (one for each component of $\theta$)

Rearranging to isolate the residuals:

$$X^T X \theta = X^T y$$

#### Solving for Parameters

Finally, we solve for $\theta$ by multiplying both sides by the inverse of $(X^T X)$:

$$\hat{\theta} = (X^T X)^{-1} X^T y$$

where:
- $(X^T X)^{-1}$: Inverse of the $p \times p$ Gram matrix $X^T X$ (exists if $X^T X$ is invertible)
- $X^T y$: Cross-product of features and targets
- $\hat{\theta}$: The estimated parameter vector (the MLE)
- This is the **closed-form solution** to our optimization problem—we can compute it directly without iteration

**Key insight**: This is exactly what we get from ordinary least squares! Maximizing likelihood under Gaussian noise is equivalent to minimizing squared error.

## 4. Estimating the Noise Variance

The MLE for the noise variance $\sigma^2$ is:

$$\hat{\sigma}^2 = \frac{1}{n}\sum_{i=1}^{n}(y_i - \hat{\theta}^T x_i)^2 = \frac{\text{RSS}}{n}$$

where:
- $\hat{\theta}$: The estimated parameters from the maximum likelihood step (the normal equation solution)
- $(y_i - \hat{\theta}^T x_i)$: Residuals (prediction errors) using the fitted model
- $\sum_{i=1}^{n}(y_i - \hat{\theta}^T x_i)^2$: RSS (Residual Sum of Squares), total squared error
- $\frac{1}{n}$: Average the RSS across all samples
- $\hat{\sigma}^2$: Estimated noise variance (how much the data varies around the fitted line)

**Important note**: This uses $n$ in the denominator (biased estimator). For unbiased estimation, use $n-p$ where $p$ is the number of parameters:

$$\text{Unbiased: } \hat{\sigma}^2 = \frac{1}{n-p}\sum_{i=1}^{n}(y_i - \hat{\theta}^T x_i)^2$$

The $n-p$ accounts for the degrees of freedom lost when estimating $p$ parameters.

## 5. Connection to Least Squares

The key insight is that **maximizing the likelihood is equivalent to minimizing the sum of squared errors**.

From the log-likelihood (ignoring constant terms):

$$\ell(\theta, \sigma^2) \propto -\frac{1}{2\sigma^2}\sum_{i=1}^{n}(y_i - \theta^T x_i)^2$$

where:
- The symbol $\propto$ means "proportional to" - we ignore terms that don't depend on $\theta$
- Maximizing this with respect to $\theta$ means finding $\theta$ that minimizes the sum of squared residuals
- The factor $-\frac{1}{2\sigma^2}$ is just a negative constant, so maximizing means minimizing the sum of squares

Maximizing $\ell$ with respect to $\theta$ is therefore equivalent to minimizing:

$$\sum_{i=1}^{n}(y_i - \theta^T x_i)^2$$

where:
- $y_i - \hat{\theta}^T x_i$: Residual for sample $i$
- $(y_i - \hat{\theta}^T x_i)^2$: Squared residual (penalizes large errors more)
- $\sum_{i=1}^{n}$: Total squared error across all samples

This is exactly the ordinary least squares (OLS) objective! Thus:

$$\boxed{\text{Maximum Likelihood Estimation under Gaussian noise} = \text{Ordinary Least Squares}}$$

## 6. Predictions and Uncertainty

### 6.1 Point Predictions

Given a new input $x_{new}$, the predicted mean (expected value of $y_{new}$) is:

$$\mathbb{E}[y_{new} | x_{new}, \hat{\theta}, \hat{\sigma}^2] = \hat{\theta}^T x_{new}$$

where:
- $\mathbb{E}[\cdot]$: Expected value (average outcome)
- $\hat{\theta}$: Fitted parameters from training
- $x_{new}$: Features for the new sample
- $\hat{\theta}^T x_{new}$: Dot product giving the predicted value

This is just our learned linear model applied to new data.

### 6.2 Prediction Intervals

#### Understanding Variance Decomposition

We know that for a prediction $\hat{y}_{new}$, the **total variance** can be decomposed into two independent sources:

1. **Aleatoric uncertainty (irreducible)**: From inherent randomness in the data ($\epsilon$ noise)
2. **Epistemic uncertainty (reducible)**: From uncertainty in parameter estimates

The general variance formula for a prediction is:

$$\text{Var}[y_{new}] = \text{Var}[\text{noise}] + \text{Var}[\text{prediction function}]$$

#### Application to Linear Regression

For our regression model, the variance of predictions includes both sources of uncertainty:

$$\text{Var}[y_{new} | x_{new}, \hat{\theta}, \hat{\sigma}^2] = \hat{\sigma}^2 + \hat{\sigma}^2 x_{new}^T(X^T X)^{-1}x_{new}$$

Breaking down the two variance terms:

**First term: $\hat{\sigma}^2$ (Noise/Aleatoric Uncertainty)**
- This is the irreducible noise variance from our model
- Always present, even with perfect parameter estimates
- Represents true randomness in the underlying process
- Same for all predictions

**Second term: $\hat{\sigma}^2 x_{new}^T(X^T X)^{-1}x_{new}$ (Parameter/Epistemic Uncertainty)**
- $(X^T X)^{-1}$: Covariance matrix of the fitted parameters; tells us how uncertain each parameter $\hat{\theta}_j$ is
- $x_{new}^T(X^T X)^{-1}x_{new}$: How much the parameter uncertainty affects this particular prediction (depends on $x_{new}$)
- Predictions far from training data have larger uncertainty
- Predictions near training data have smaller uncertainty
- This uncertainty reduces as we collect more data

**Key insight**:
- Predictions at the center of the training data: small epistemic term, mainly noise uncertainty
- Predictions far from training data: large epistemic term, high extrapolation uncertainty
- The epistemic term goes to zero as $n \to \infty$ (more data = less parameter uncertainty)
- The aleatoric term remains (fundamental randomness in the process)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
from scipy.optimize import minimize

# Set random seed for reproducibility
np.random.seed(42)

## 7. Example: 1D Linear Regression

In [ ]:
# Generate synthetic data
n_samples = 50
X = np.random.randn(n_samples, 1)
true_theta = np.array([2.5])
true_sigma = 0.5

# Generate target with noise
y = X @ true_theta + np.random.randn(n_samples) * true_sigma

# Add bias term (intercept)
X_design = np.hstack([np.ones((n_samples, 1)), X])

# Solve normal equations
theta_hat = np.linalg.inv(X_design.T @ X_design) @ X_design.T @ y

# Estimate noise variance
residuals = y - X_design @ theta_hat
sigma_hat_sq = np.sum(residuals**2) / n_samples
sigma_hat = np.sqrt(sigma_hat_sq)

print(f"Estimated intercept: {theta_hat[0]:.4f}")
print(f"Estimated slope: {theta_hat[1]:.4f} (true: {true_theta[0]:.4f})")
print(f"Estimated noise std: {sigma_hat:.4f} (true: {true_sigma:.4f})")

## 8. Visualizing the Model and Uncertainty

In [ ]:
# Create prediction grid
x_test = np.linspace(X.min() - 1, X.max() + 1, 200).reshape(-1, 1)
x_test_design = np.hstack([np.ones((x_test.shape[0], 1)), x_test])

# Predictions
y_pred = x_test_design @ theta_hat

# Compute prediction intervals
# Variance includes model uncertainty and noise
X_var = np.diag(x_test_design @ np.linalg.inv(X_design.T @ X_design) @ x_test_design.T)
pred_std = np.sqrt(sigma_hat_sq * (1 + X_var))

# Confidence interval for the mean
mean_std = np.sqrt(sigma_hat_sq * X_var)

# Plot
plt.figure(figsize=(10, 6))

# Data points
plt.scatter(X, y, alpha=0.6, s=50, label='Observations')

# Mean prediction
plt.plot(x_test, y_pred, 'r-', linewidth=2, label='Mean prediction')

# Confidence interval for mean (95%)
ci_mean = 1.96 * mean_std
plt.fill_between(x_test.flatten(), (y_pred - ci_mean).flatten(), (y_pred + ci_mean).flatten(),
                  alpha=0.3, color='red', label='95% CI (mean)')

# Prediction interval (95%)
pi = 1.96 * pred_std
plt.fill_between(x_test.flatten(), (y_pred - pi).flatten(), (y_pred + pi).flatten(),
                  alpha=0.2, color='blue', label='95% PI (single obs)')

plt.xlabel('x')
plt.ylabel('y')
plt.title('Probabilistic Linear Regression: Mean and Uncertainty')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 9. Likelihood Function Visualization

In [ ]:
# Compute log-likelihood for different parameter values
slope_range = np.linspace(1.5, 3.5, 100)
intercept_range = np.linspace(-1, 1, 100)
log_likelihood = np.zeros((len(intercept_range), len(slope_range)))

for i, intercept in enumerate(intercept_range):
    for j, slope in enumerate(slope_range):
        theta = np.array([intercept, slope])
        residuals = y - X_design @ theta
        # Log-likelihood (up to constant)
        log_likelihood[i, j] = -np.sum(residuals**2) / (2 * sigma_hat_sq)

# Plot
plt.figure(figsize=(10, 8))
contour = plt.contour(slope_range, intercept_range, log_likelihood, levels=20, cmap='viridis')
plt.clabel(contour, inline=True, fontsize=8)
plt.plot(theta_hat[1], theta_hat[0], 'r*', markersize=15, label='MLE')
plt.xlabel('Slope')
plt.ylabel('Intercept')
plt.title('Log-Likelihood Surface')
plt.colorbar(contour, label='Log-Likelihood')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 10. Bayesian Perspective

### 10.1 Bayes' Rule for Regression

#### The Fundamental Bayes' Rule

We know that **Bayes' rule** is a fundamental formula in probability theory that describes how to update our beliefs given new evidence:

$$p(A | B) = \frac{p(B | A) p(A)}{p(B)}$$

This relates:
- **Posterior** $p(A | B)$: Probability of $A$ after observing $B$
- **Likelihood** $p(B | A)$: Probability of observing $B$ if $A$ is true
- **Prior** $p(A)$: Probability of $A$ before observing $B$
- **Evidence** $p(B)$: Total probability of observing $B$

In words: **Posterior = (Likelihood × Prior) / Evidence**

#### Application to Regression

Applying Bayes' rule to our regression problem where we want to infer $\theta$ given data $(X, y)$:

$$p(\theta | X, y) = \frac{p(y | X, \theta) p(\theta)}{p(y | X)}$$

where:
- $p(\theta | X, y)$: The **posterior** distribution - probability of parameters $\theta$ given observed data $(X, y)$. This is what we want to infer (our updated belief about $\theta$).
- $p(y | X, \theta)$: The **likelihood** - probability of observing data $y$ given features $X$ and parameters $\theta$ (we derived this in section 3)
- $p(\theta)$: The **prior** distribution - our belief about $\theta$ before seeing any data
- $p(y | X)$: The **marginal likelihood** or **evidence** - total probability of observing the data averaged over all possible parameters
- The denominator $p(y | X) = \int p(y | X, \theta) p(\theta) d\theta$ integrates the likelihood-times-prior over all possible $\theta$ values to get a normalizing constant

**Interpretation**: The posterior is proportional to "likelihood times prior"—the data (via the likelihood) updates our prior beliefs about the parameters.

### 10.2 Gaussian Prior and Posterior

#### Conjugate Priors

We know that when both the likelihood and prior are Gaussian distributions, the posterior is also Gaussian. This is called a **conjugate prior** relationship—the posterior has the same functional form as the prior. This makes Bayesian inference tractable (analytically solvable).

#### Setting Up the Prior

If we use a Gaussian prior (expressing our belief that $\theta$ clusters around $\mu_0$ with uncertainty $\Sigma_0$):

$$p(\theta) = \mathcal{N}(\mu_0, \Sigma_0)$$

where:
- $\mu_0$: Prior mean (our best guess for $\theta$ before seeing data)
- $\Sigma_0$: Prior covariance (how confident we are in $\mu_0$)

#### Deriving the Posterior

Given our Gaussian likelihood from section 3 and this Gaussian prior, the posterior is also Gaussian:

$$p(\theta | X, y) = \mathcal{N}(\mu_n, \Sigma_n)$$

where the posterior parameters combine information from the prior and the data:

$$\Sigma_n = (\Sigma_0^{-1} + \frac{1}{\sigma^2}X^T X)^{-1}$$

Breaking down this formula:
- $\Sigma_0^{-1}$: **Prior precision** (inverse of prior covariance) - how much we trust the prior
- $\frac{1}{\sigma^2}X^T X$: **Data precision** - how much information the data provides (more samples or lower noise = higher precision)
- The sum $\Sigma_0^{-1} + \frac{1}{\sigma^2}X^T X$: Total precision (prior + data)
- Taking the inverse gives $\Sigma_n$: **Posterior covariance** - combines prior and data uncertainty

$$\mu_n = \Sigma_n (\Sigma_0^{-1}\mu_0 + \frac{1}{\sigma^2}X^T y)$$

Breaking down this formula:
- $\Sigma_0^{-1}\mu_0$: **Prior information** (prior precision weighted by prior mean)
- $\frac{1}{\sigma^2}X^T y$: **Data information** (precision-weighted data)
- $\mu_n$: **Posterior mean** - a weighted average of prior and data, where weights are determined by precisions

**Key insight**: Posterior = precision-weighted average of prior and data. More confident priors (high $\Sigma_0^{-1}$) dominate; better data (high precision) dominates.

#### Special Case: No Prior (Diffuse Prior)

With a diffuse (uninformative) prior—meaning we're very uncertain before seeing data ($\Sigma_0 \to \infty$, so $\Sigma_0^{-1} \to 0$):

$$\lim_{\Sigma_0 \to \infty} \mu_n = (X^T X)^{-1} X^T y = \hat{\theta}_{\text{MLE}}$$

The posterior mean becomes the MLE! This shows that **maximum likelihood estimation is a special case of Bayesian inference with an uninformative prior**. The difference only appears when we have informative priors or small datasets.

## 11. Practical Implications

### 11.1 When to Use This Framework

- **Uncertainty quantification**: Get confidence/prediction intervals automatically
- **Hypothesis testing**: Test if parameters are significantly different from zero
- **Model comparison**: Use likelihood ratios or information criteria
- **Regularization**: Priors naturally lead to ridge/lasso regression

### 11.2 Key Assumptions

1. **Linearity**: The relationship between $x$ and $y$ is linear
2. **Constant variance**: Noise variance $\sigma^2$ is constant (homoscedasticity)
3. **Independence**: Observations are independent
4. **Gaussian noise**: Errors are normally distributed

Violations of these assumptions can affect inference validity.

## 12. Summary

### Quick Reference Table

| Aspect | Formula/Value | Explanation |
|--------|---------------|-------------|
| **Model** | $p(y \mid x, \theta, \sigma^2) = \mathcal{N}(\theta^T x, \sigma^2)$ | Target is normally distributed around linear prediction |
| **Likelihood** | $L(\theta, \sigma^2) = \prod_i \mathcal{N}(y_i; \theta^T x_i, \sigma^2)$ | Probability of all data given parameters |
| **Log-Likelihood** | $\ell = -\frac{n}{2}\log(2\pi\sigma^2) - \frac{1}{2\sigma^2}\sum_i (y_i - \hat{\theta}^T x_i)^2$ | Log of likelihood; easier to optimize |
| **MLE for $\theta$** | $\hat{\theta} = (X^T X)^{-1} X^T y$ | Parameters that maximize likelihood (Normal Equation) |
| **MLE for $\sigma^2$** | $\hat{\sigma}^2 = \frac{1}{n}\sum_i (y_i - \hat{\theta}^T x_i)^2$ | Estimated noise variance (RSS/n) |
| **Connection to OLS** | Maximizing $L(\theta)$ ⟺ Minimizing $\sum(y_i - \hat{\theta}^T x_i)^2$ | MLE under Gaussian noise = Least Squares |
| **Point Prediction** | $\hat{y}_{new} = \hat{\theta}^T x_{new}$ | Expected value of new target |
| **Prediction Variance** | $\text{Var}[y_{new}] = \hat{\sigma}^2(1 + x_{new}^T(X^TX)^{-1}x_{new})$ | Uncertainty = noise + parameter uncertainty |
| **Confidence Int. (mean)** | $\hat{y} \pm 1.96\sqrt{\hat{\sigma}^2 x_{new}^T(X^TX)^{-1}x_{new}}$ | 95% CI for mean prediction (no noise term) |
| **Prediction Int.** | $\hat{y} \pm 1.96\sqrt{\hat{\sigma}^2(1 + x_{new}^T(X^TX)^{-1}x_{new})}$ | 95% PI for individual observation (includes noise) |
| **Bayesian Prior** | $p(\theta) = \mathcal{N}(\mu_0, \Sigma_0)$ | Prior belief about parameters before seeing data |
| **Bayesian Posterior** | $p(\theta \mid X, y) = \mathcal{N}(\mu_n, \Sigma_n)$ | Updated belief about parameters after data |

### Key Takeaways

1. **Linear Regression = MLE under Gaussian noise**: The familiar least squares solution emerges naturally from the probabilistic model
2. **Uncertainty Quantification**: The framework automatically provides confidence and prediction intervals
3. **Two Sources of Uncertainty**:
   - Aleatoric (data noise): $\sigma^2$ - irreducible, present even with perfect parameter estimates
   - Epistemic (model): $x_{new}^T(X^TX)^{-1}x_{new}$ - reduced as we gather more data
4. **Bayesian Generalization**: Adding priors leads to ridge regression, lasso regression, and other regularization methods
5. **Connection to Modern ML**: Understanding this foundation is crucial for Bayesian neural networks, Gaussian processes, and other probabilistic models

## References

- Bishop, C. M. (2006). Pattern Recognition and Machine Learning. Springer.
- Murphy, K. P. (2012). Machine Learning: A Probabilistic Perspective. MIT Press.
- James, G., Witten, D., Hastie, T., & Tibshirani, R. (2013). An Introduction to Statistical Learning. Springer.

---
## Practice Exercises

**Conceptual**

1. We modelled $y = \theta^T x + \epsilon$ with $\epsilon \sim \mathcal{N}(0, \sigma^2)$. What probability distribution does this impose on $y \mid x$, and what are its mean and variance?

2. Maximising $L(\theta, \sigma^2)$ over the entire dataset gives $\hat\theta = (X^TX)^{-1}X^Ty$. Re-derive this by differentiating the log-likelihood and setting the gradient to zero. (Show all steps.)

3. The MLE for $\sigma^2$ is $\hat\sigma^2 = \frac{1}{n}\sum_i(y_i - \hat\theta^T x_i)^2$. Why is this called a *biased* estimator? What denominator would make it unbiased?

4. In the variance decomposition $\text{Var}[y_\text{new}] = \hat\sigma^2 + \hat\sigma^2 x_\text{new}^T(X^TX)^{-1}x_\text{new}$, which term represents aleatoric uncertainty and which represents epistemic uncertainty? Which goes to zero as $n \to \infty$?

5. With Gaussian prior $p(\theta) = \mathcal{N}(\mu_0, \Sigma_0)$, the posterior mean is $\mu_n = \Sigma_n(\Sigma_0^{-1}\mu_0 + \frac{1}{\sigma^2}X^Ty)$. Show that as $\Sigma_0 \to \infty$ (diffuse prior), $\mu_n \to \hat\theta_\text{MLE}$.

**Numerical**

6. Generate $n=50$ samples from $y = 3x + 1 + \epsilon$, $\epsilon \sim \mathcal{N}(0, 0.5^2)$, $x \sim \mathcal{N}(0,1)$. Compute $\hat\theta$ via the Normal Equation. How close are the estimates to the true values?

7. For the model above, compute the 95% prediction interval for a new point $x_\text{new} = 2$. Plot the data, the fitted line, and the interval.

**Reflection**

8. You have 10 data points and 8 features. What issue arises with the Normal Equation? How does the Bayesian approach handle this naturally?